<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/module34a/Lab01.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Lab 1 — From Java/C++ to Qubits
### Python, NumPy, and Complex Amplitudes

**Course:** Quantum Optimization and Simulation
**Maps to:** Module 3, Lessons 1–2 (Hydrogen atom / molecule, ground state energy)

**Time:** ~60 minutes on your own (instructor walkthrough ~12 min)

---

### Why this lab exists

Everything in Modules 3 and 4 rests on one idea: a molecule's state is a **weighted mix
of configurations**, and the weights are *complex numbers* whose squared magnitudes are
probabilities. Before we build any quantum circuit, we make that concrete in Python.

### After this lab you can
1. Run and edit a Jupyter/Colab notebook.
2. Translate the Python you need from your C++/Java instincts.
3. Represent a qubit state as a NumPy vector and read off probabilities.
4. Apply gates as matrices, and check that a matrix is a legal quantum operator.
5. Build the same state in Qiskit, run it on the Aer simulator, and see that the
   measured counts match $|amplitude|^2$.
6. Write the "mix of bonding and antibonding" state that Module 3 is really about.

## 0. Notebook survival guide (2 minutes)

A notebook is a list of **cells**. A cell is either text (like this one) or code.

* **Run a cell:** `Shift+Enter`.
* Cells share **one** namespace, top to bottom. A variable defined in cell 3 is visible
  in cell 9 — but only if you actually ran cell 3. If something is "not defined,"
  you probably skipped a cell.
* **Restart & Run All** (Kernel menu) is the fix for 90% of confusing states.
* The last expression in a cell prints automatically — no `print` needed.

Run the cell below to install the packages (Colab) and check your versions.

In [ ]:
# On Colab, uncomment the next line the first time you open this notebook.
# %pip install -q qiskit qiskit-aer matplotlib

import qiskit, qiskit_aer, numpy as np
print("qiskit    ", qiskit.__version__)
print("qiskit-aer", qiskit_aer.__version__)
print("numpy     ", np.__version__)

## 1. Python for the C++/Java programmer

You already know loops, functions, and arrays. Here is the 90% you need, side by side.

| You are used to | Python |
|---|---|
| `int n = 5;` | `n = 5`  (no type, no semicolon) |
| `for (int i=0;i<n;i++)` | `for i in range(n):` |
| `{ }` blocks | **indentation** (4 spaces) |
| `double[] a = new double[3];` | `a = np.zeros(3)` |
| `a.length` | `len(a)` |
| `Math.sqrt(x)` | `np.sqrt(x)` |
| `//` comment | `#` comment |
| `System.out.println(x)` | `print(x)` |
| complex numbers: a library | built in: `2 + 3j` |

Two Python habits worth stealing right away:

```python
squares = [i*i for i in range(5)]      # list comprehension -> [0,1,4,9,16]
a, b = b, a                            # swap, no temp variable
```

And the one that matters most here: **NumPy arrays do arithmetic elementwise, and
`@` means matrix multiplication.**

In [ ]:
import numpy as np

a = np.array([1.0, 2.0, 3.0])
M = np.array([[0, 1],
              [1, 0]])
v = np.array([1, 0])

print("a * 2      =", a * 2)          # elementwise
print("a + a      =", a + a)
print("M @ v      =", M @ v)          # matrix-vector product  (NOT M * v)
print("M * M      =\n", M * M)        # elementwise -- almost never what you want
print("M @ M      =\n", M @ M)        # real matrix product

## 2. Complex numbers, only as much as we need

Python writes the imaginary unit as `j`, not `i`. You will **not** be asked to multiply
complex numbers by hand in this course — NumPy does it. You only need three facts:

1. A complex number $z = x + iy$ has a **magnitude** $|z| = \sqrt{x^2+y^2}$ and a
   **phase** (an angle). Think of it as an arrow in the plane.
2. $|z|^2 = z^*z$ is a **real, non-negative** number. This is what becomes a probability.
3. $e^{i\phi}$ is an arrow of length 1 pointing at angle $\phi$. Multiplying by it
   **rotates without changing the length** — so it never changes probabilities.

Fact 3 is why the whole course talks about "phase rotation": phases are invisible to a
single measurement, but they decide how amplitudes **add up** when configurations mix.

In [ ]:
z = 3 + 4j
print("z          =", z)
print("|z|        =", abs(z))              # 5.0
print("|z|^2      =", abs(z)**2)           # 25.0
print("conjugate  =", np.conj(z))

phi = np.pi / 3
u = np.exp(1j * phi)
print("\ne^(i*pi/3) =", np.round(u, 4), " with |u| =", np.round(abs(u), 12))
print("multiplying z by u keeps |z|:", np.round(abs(z * u), 12))

### Exercise 1 — constructive vs destructive addition

Module 3 says two electron configurations combine "constructively" (bonding) or
"destructively" (antibonding). That is just complex addition.

Fill in the TODO: return the squared magnitude $|z_1 + z_2|^2$.

In [ ]:
def combined_intensity(z1, z2):
    '''Return |z1 + z2|^2.'''
    # TODO: one line
    ...

same_phase     = combined_intensity(1+0j,  1+0j)    # constructive
opposite_phase = combined_intensity(1+0j, -1+0j)    # destructive

print("constructive:", same_phase)      # expect 4.0
print("destructive :", opposite_phase)  # expect 0.0
assert np.isclose(same_phase, 4.0) and np.isclose(opposite_phase, 0.0)
print("PASS")

Two arrows of length 1 give intensity **4** when aligned and **0** when opposed — not 2
and 2. That factor is the entire difference between a bond forming and not forming.

## 3. A qubit is a 2-vector

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle
\quad\longleftrightarrow\quad
\begin{pmatrix}\alpha\\ \beta\end{pmatrix},
\qquad |\alpha|^2 + |\beta|^2 = 1 .$$

$\alpha,\beta$ are **amplitudes** (complex). $|\alpha|^2$ is the probability of reading
0. The normalization condition is just "probabilities sum to 1."

In [ ]:
ket0 = np.array([1, 0], dtype=complex)
ket1 = np.array([0, 1], dtype=complex)

def probabilities(psi):
    '''Return the measurement probabilities of a state vector.'''
    return np.abs(psi)**2

psi = (ket0 + ket1) / np.sqrt(2)          # equal superposition
print("psi   =", np.round(psi, 4))
print("probs =", np.round(probabilities(psi), 4), " sum =", probabilities(psi).sum())

### Exercise 2 — normalize, and see that global phase is invisible

In [ ]:
def normalize(v):
    '''Scale a vector to unit length.  Hint: np.linalg.norm'''
    v = np.asarray(v, dtype=complex)
    # TODO
    ...

psi = normalize([3, 4])
print("normalized  :", np.round(psi, 4))
print("probabilities:", np.round(probabilities(psi), 4))   # expect [0.36, 0.64]
assert np.isclose(np.linalg.norm(psi), 1.0)

# Multiply the WHOLE state by a phase -> nothing observable changes.
psi_rotated = np.exp(1j * 1.234) * psi
print("after global phase:", np.round(probabilities(psi_rotated), 4))
assert np.allclose(probabilities(psi), probabilities(psi_rotated))
print("PASS")

## 4. Gates are matrices — and they must be *unitary*

A gate $U$ is legal only if $U^\dagger U = I$, where $\dagger$ means
"transpose **and** conjugate." Physically: it must map unit-length vectors to
unit-length vectors, so total probability stays 1.

This is exactly the check Module 3 applies when it asks whether $X\otimes X + Y\otimes Y$
is "a legitimate quantum operator." (Spoiler for Lab 2: it is not.)

In [ ]:
I2 = np.eye(2, dtype=complex)
X  = np.array([[0, 1], [1, 0]], dtype=complex)
Z  = np.array([[1, 0], [0, -1]], dtype=complex)
H  = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)

def is_unitary(U, tol=1e-10):
    U = np.asarray(U, dtype=complex)
    return np.allclose(U.conj().T @ U, np.eye(U.shape[0]), atol=tol)

for name, M in [("X", X), ("Z", Z), ("H", H), ("X+Z (not a gate)", X + Z)]:
    print(f"{name:18s} unitary? {is_unitary(M)}")

### Exercise 3 — apply H by hand, then check the probabilities

Compute $H|0\rangle$ and $H|1\rangle$ with a matrix-vector product.
Both should give 50/50 probabilities — but **different phases**, which is the part that
matters later.

In [ ]:
h0 = ...   # TODO: H applied to |0>
h1 = ...   # TODO: H applied to |1>

print("H|0> =", np.round(h0, 4), " probs:", np.round(probabilities(h0), 4))
print("H|1> =", np.round(h1, 4), " probs:", np.round(probabilities(h1), 4))
assert np.allclose(probabilities(h0), [0.5, 0.5])
assert np.allclose(probabilities(h1), [0.5, 0.5])
assert not np.allclose(h0, h1)          # same probabilities, different states!
print("PASS -- identical probabilities, different amplitudes (the sign on |1>).")

## 5. The same thing in Qiskit

Qiskit gives you circuits instead of raw matrices. Two tools we will use all course:

* `Statevector(qc)` — the exact amplitudes (a simulator luxury; hardware never gives you this).
* `AerSimulator` — sampling with a finite number of **shots**, like real hardware.

> ### ⚠️ Bit ordering — read this once, remember it forever
> Qiskit prints bitstrings **right to left**: the printed string is
> $q_{n-1}\dots q_1 q_0$. So the printed `'01'` means **$q_0 = 1$, $q_1 = 0$**.
> The lecture slides write $|q_0 q_1\rangle$ left to right. The strings still *look*
> the same for the states we care about, but when you index a Python string, remember
> that `bits[::-1]` reverses it into qubit order. We will use this constantly in Lab 4.

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator

qc = QuantumCircuit(1)
qc.h(0)
print(qc.draw(output="text"))

sv = Statevector(qc)
print("\namplitudes  :", np.round(sv.data, 4))
print("probabilities:", sv.probabilities_dict())

In [ ]:
sim = AerSimulator()

qc_m = QuantumCircuit(1)
qc_m.h(0)
qc_m.measure_all()

counts = sim.run(transpile(qc_m, sim), shots=1000).result().get_counts()
print("1000 shots:", counts)
print("fractions :", {k: v/1000 for k, v in counts.items()})

### Exercise 4 — the "mix" state, one lab early

Module 3's punchline is that the true ground state is

$$|\Psi(\theta)\rangle \;=\; \cos\theta\,|\text{bonding}\rangle \;+\; \sin\theta\,|\text{antibonding}\rangle ,$$

with a **small but nonzero** $\sin\theta$. Build exactly that state on one qubit
(`|0>` = bonding, `|1>` = antibonding) using the `ry` gate, which rotates by
$\cos(\phi/2), \sin(\phi/2)$ — note the factor of 2.

In [ ]:
def mix_state(theta):
    '''Return the Statevector cos(theta)|0> + sin(theta)|1>.
    Hint: qc.ry(angle, 0) produces cos(angle/2)|0> + sin(angle/2)|1>.'''
    qc = QuantumCircuit(1)
    # TODO: one line
    ...
    return Statevector(qc)

for theta in [0.0, 0.112, np.pi/4]:
    sv = mix_state(theta)
    print(f"theta={theta:6.3f}   amplitudes={np.round(sv.data,4)}   "
          f"P(antibonding)={sv.probabilities()[1]:.4f}")

sv = mix_state(0.112)
assert np.allclose(sv.data, [np.cos(0.112), np.sin(0.112)], atol=1e-9)
print("\nPASS")

### Exercise 5 — shots are noisy; that is the whole cost story of VQE

Estimate $P(1)$ for $\theta = 0.112$ by sampling, at several shot counts, and watch the
error shrink like $1/\sqrt{N}$.

In [ ]:
import matplotlib.pyplot as plt

theta = 0.112
exact_p1 = np.sin(theta)**2

qc = QuantumCircuit(1)
qc.ry(2*theta, 0)
qc.measure_all()
tqc = transpile(qc, sim)

shot_list = [100, 400, 1600, 6400, 25600]
errors = []
for n in shot_list:
    counts = sim.run(tqc, shots=n, seed_simulator=7).result().get_counts()
    p1 = counts.get("1", 0) / n
    errors.append(abs(p1 - exact_p1))
    print(f"{n:6d} shots -> P(1) = {p1:.5f}   |error| = {abs(p1-exact_p1):.5f}")

plt.figure(figsize=(5,3.2))
plt.loglog(shot_list, errors, "o-", label="measured error")
plt.loglog(shot_list, [errors[0]*np.sqrt(shot_list[0]/n) for n in shot_list],
           "k--", label=r"$1/\sqrt{N}$ reference")
plt.xlabel("shots"); plt.ylabel("|error|"); plt.legend(); plt.tight_layout(); plt.show()

## 6. Checkpoint — answer these before Lab 2

1. A state has amplitudes $(0.6,\;0.8i)$. What are the two probabilities? Is the state
   normalized?
2. Two configurations each have amplitude $0.5$. What is the probability of the combined
   outcome if they add **in phase**? **Out of phase**? Why is neither answer $0.5$?
3. Why is $X + Z$ not a valid gate, even though $X$ and $Z$ both are?
4. Qiskit prints the counts `{'10': 512, '00': 488}` for a 2-qubit circuit.
   Which qubit is the one that is sometimes 1?
5. To halve the statistical error of a measured expectation value, by what factor must
   you increase the number of shots? (This single fact explains the
   "1,000,000,000 shots" slide at the end of Module 3.)

### What is next
**Lab 2** builds the two-qubit Pauli operators and the `RZZ` gate — the first real
circuit ingredient of the VQE ansatz.